# Model Experiment: DLinear

**ამოცანა:** Walmart Store Sales Forecasting — Deep Learning არქიტექტურა


## რატომ DLinear?

**თეორია:** DLinear ეყრდნობა კლასიკურ **decomposition** იდეას (ისევე როგორც STL/Prophet),
მაგრამ ძალიან მარტივი architecture-ით:

1. **Series Decomposition** — შემავალი სერია იშლება ორ ნაწილად moving-average-ის
   საშუალებით:
   - **Trend** — moving average-ის შედეგი (გლუვი, ნელა ცვალებადი კომპონენტი)
   - **Seasonal/Residual** — `original - trend` (სწრაფად ცვალებადი, სეზონური რხევები)
2. **Linear mapping** — თითოეულ კომპონენტზე ცალკე **ერთი Linear layer** გადადის
   `lookback_window → forecast_horizon`-ზე (წონების matrix, bias-ის გარეშე ან
   მარტივი bias-ით — არავითარი attention, RNN, ან non-linearity)
3. **შეჯამება** — საბოლოო პროგნოზი = trend_forecast + seasonal_forecast

**რატომ მუშაობს ეს ასე კარგად:** ავტორების არგუმენტია, რომ self-attention-ის
**permutation-invariant** ბუნება კარგავს დროით (temporal) ინფორმაციას, რაც
დროით სერიებში კრიტიკულია — უბრალო linear mapping კი ამ ინფორმაციას ინარჩუნებს.


**მთავარი პრაქტიკული უპირატესობა:** რადგან DLinear-ის წონები **გაზიარებულია** ყველა
store-dept სერიას შორის (channel-independent, მაგრამ shared weights), შესაძლებელია
**ერთი მოდელის** დატრენინგება მთელს დატასეტზე ერთდროულად — ეს პირდაპირ პასუხობს
ARIMA/SARIMA-ს მთავარ შეზღუდვას (რომ თითოეულ სერიას ცალკე tuning სჭირდებოდა).


In [ ]:
!pip install wandb -q



In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### `data_prep.py`-ის ფუნქციები (inline)

იგივე ვერსია რაც წინა notebook-ებშია — Kaggle-ის environment-ს არ აქვს წვდომა
repo-ს `src/` დირექტორიაზე.

In [ ]:
def merge_all(sales_df, features_df, stores_df):
    df = sales_df.merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
    df = df.merge(stores_df, on="Store", how="left")

    if "IsHoliday_feat" in df.columns:
        df = df.drop(columns=["IsHoliday_feat"])

    return df


def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    is_holiday = np.asarray(is_holiday)

    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()

WANDB_PROJECT = "walmart-forecasting-statistical"


## 1. მონაცემების ჩატვირთვა

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

def load_raw_data_kaggle(data_dir=KAGGLE_DATA_DIR):
    train = pd.read_csv(f"{data_dir}/train.csv.zip")
    test = pd.read_csv(f"{data_dir}/test.csv.zip")
    features = pd.read_csv(f"{data_dir}/features.csv.zip")
    stores = pd.read_csv(f"{data_dir}/stores.csv")

    train["Date"] = pd.to_datetime(train["Date"])
    test["Date"] = pd.to_datetime(test["Date"])
    features["Date"] = pd.to_datetime(features["Date"])

    return train, test, features, stores


train, test, features, stores = load_raw_data_kaggle()
df = merge_all(train, features, stores)
df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
df["series_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)

print(f"Total rows: {len(df)}, unique store-dept series: {df['series_id'].nunique()}")


## 2. Panel აგება და sliding windows

DLinear-ს სჭირდება ფიქსირებული ზომის `(lookback_window → forecast_horizon)` წყვილები.
ვაგებთ **panel**-ს (store-dept × date მატრიცას) და ვაწარმოებთ sliding window sample-ებს
ყველა საკმარისი სიგრძის სერიიდან — ეს ტრენინგის მონაცემია **ყველა** სერიისთვის
ერთდროულად, რაც ARIMA/SARIMA-სგან განსხვავებული, scalable მიდგომაა.

- **`LOOKBACK = 52`** კვირა (ერთი წელი ისტორია)
- **`HORIZON = 13`** კვირა (იგივე რაც წინა notebook-ებში გამოვიყენეთ შედარებადობისთვის)
- **`MAX_SERIES`** — შეზღუდვა სწრაფი იტერაციისთვის Kaggle-ზე; `None`-ზე დააყენეთ სრული
  დატასეტისთვის (3000+ სერია — DLinear მსუბუქია, სავარაუდოდ მაინც სწრაფად დაასრულებს)

In [ ]:
LOOKBACK = 52
HORIZON = 13
MAX_SERIES = 300  # None ყველა სერია.300 სწრაფი იტერაციისთვის, საკმარისია დემონსტრირებისთვის.

def build_windows(raw_df, lookback=LOOKBACK, horizon=HORIZON, max_series=MAX_SERIES, val_horizon=HORIZON):
    """
    აბრუნებს train/val sliding-window sample-ებს ყველა (ან შერჩეულ) store-dept სერიაზე.
    Val არის დროში ბოლო `val_horizon` კვირა თითოეულ სერიაზე (გლობალური time-based split).
    ასევე აბრუნებს val_series_ids-ს — ერთი series_id თითო val window-ზე, რომ მოგვიანებით
    per-series WMAE breakdown გამოვთვალოთ (და ARIMA/SARIMA/Prophet-თან პირდაპირ შევადაროთ).
    """
    series_ids = raw_df["series_id"].unique()
    if max_series is not None:
        series_ids = series_ids[:max_series]

    X_train, y_train, holiday_train = [], [], []
    X_val, y_val, holiday_val, val_series_ids = [], [], [], []

    for sid in series_ids:
        sub = raw_df[raw_df["series_id"] == sid].sort_values("Date")
        sales = sub["Weekly_Sales"].values.astype(np.float32)
        holidays = sub["IsHoliday"].values.astype(bool)

        n = len(sales)
        if n < lookback + horizon + val_horizon:
            continue  # არასაკმარისი ისტორია ამ სერიისთვის

        cutoff = n - val_horizon

        # train windows: ყველა sliding window cutoff-მდე 
        for end in range(lookback, cutoff - horizon + 1):
            X_train.append(sales[end - lookback:end])
            y_train.append(sales[end:end + horizon])
            holiday_train.append(holidays[end:end + horizon])

        # val window: ერთი, ბოლო lookback -> ბოლო horizon
        val_start = cutoff - lookback
        if val_start >= 0:
            X_val.append(sales[val_start:cutoff])
            y_val.append(sales[cutoff:cutoff + horizon])
            holiday_val.append(holidays[cutoff:cutoff + horizon])
            val_series_ids.append(sid)

    return (
        np.array(X_train), np.array(y_train), np.array(holiday_train),
        np.array(X_val), np.array(y_val), np.array(holiday_val),
        val_series_ids,
    )


X_train, y_train, h_train, X_val, y_val, h_val, val_series_ids = build_windows(df)
print(f"Train windows: {X_train.shape}, Val windows: {X_val.shape}")


## 3. DLinear არქიტექტურა

**Series Decomposition:** moving average (`kernel_size`) ცალკე ხსნის trend-ს,
`residual = x - trend` არის seasonal ნაწილი.

**ორი დამოუკიდებელი Linear layer:** `Linear(lookback, horizon)` — ერთი trend-ისთვის,
ერთი seasonal-ისთვის. წონები **გაზიარებულია** ყველა სერიას შორის (ეს არის
"shared" ვარიანტი DLinear paper-ის ტერმინოლოგიით — ავტორები გვთავაზობენ ასევე
"individual" ვარიანტს ცალკეული წონებით თითო channel-ზე, მაგრამ ჩვენი მიზნისთვის —
ერთი scalable გლობალური მოდელი — shared წონები ბუნებრივი არჩევანია).

In [ ]:
class SeriesDecomposition(nn.Module):
    """Moving-average based trend/seasonal decomposition."""
    def __init__(self, kernel_size=25):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg_pool = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        # x: (batch, lookback)
        pad_left = (self.kernel_size - 1) // 2
        pad_right = self.kernel_size - 1 - pad_left
        front = x[:, :1].repeat(1, pad_left)
        end = x[:, -1:].repeat(1, pad_right)
        x_padded = torch.cat([front, x, end], dim=1)  # (batch, lookback + kernel - 1)

        trend = self.avg_pool(x_padded.unsqueeze(1)).squeeze(1)  # (batch, lookback)
        seasonal = x - trend
        return trend, seasonal


class DLinear(nn.Module):
    def __init__(self, lookback, horizon, kernel_size=25):
        super().__init__()
        self.decomposition = SeriesDecomposition(kernel_size)
        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        # x: (batch, lookback)
        trend, seasonal = self.decomposition(x)
        trend_out = self.linear_trend(trend)
        seasonal_out = self.linear_seasonal(seasonal)
        return trend_out + seasonal_out  # (batch, horizon)


## 4. Dataset, Loss (WMAE surrogate) და ტრენინგის მომზადება

Normalization მნიშვნელოვანია — Weekly_Sales მასშტაბი store/dept-ის მიხედვით ძალიან
განსხვავდება (ზოგან რამდენიმე ასეული, ზოგან ასეულ ათასამდე), ხოლო DLinear-ის წონები
გაზიარებულია ყველა სერიაზე. ამიტომ თითოეულ window-ს **ცალკე ვანორმალიზებთ** მისივე
lookback-ის mean/std-ით (instance normalization — ეს პრაქტიკაც DLinear/PatchTST
literature-შია სტანდარტული).

**Loss ფუნქცია** — პირდაპირ ვიყენებთ **weighted MAE**-ს (WMAE-ს დიფერენცირებად
ვერსიას) `MSE`-ის ნაცვლად, რომ ტრენინგის დროსვე ოპტიმიზაცია target metric-თან
შეესაბამებოდეს (holiday კვირებს წონა 5).

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y, is_holiday):
        self.X = X
        self.y = y
        self.is_holiday = is_holiday

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        h = self.is_holiday[idx].astype(np.float32)

        mean, std = x.mean(), x.std() + 1e-6
        x_norm = (x - mean) / std
        y_norm = (y - mean) / std

        return (
            torch.tensor(x_norm, dtype=torch.float32),
            torch.tensor(y_norm, dtype=torch.float32),
            torch.tensor(h, dtype=torch.float32),
            torch.tensor(mean, dtype=torch.float32),
            torch.tensor(std, dtype=torch.float32),
        )


def weighted_mae_loss(y_pred, y_true, is_holiday, holiday_weight=5.0):
    weights = torch.where(is_holiday > 0.5, holiday_weight, 1.0)
    return (weights * torch.abs(y_pred - y_true)).sum() / weights.sum()


train_dataset = WindowDataset(X_train, y_train, h_train)
val_dataset = WindowDataset(X_val, y_val, h_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 5. ტრენინგის loop + W&B ლოგირება

In [ ]:
KERNEL_SIZE = 25
EPOCHS = 30
LEARNING_RATE = 1e-3
PATIENCE = 5  # early stopping — თუ val_loss PATIENCE ზედიზედ epoch-ში არ გაუმჯობესდა, ვჩერდებით

model = DLinear(lookback=LOOKBACK, horizon=HORIZON, kernel_size=KERNEL_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

run = wandb.init(project=WANDB_PROJECT, name="DLinear_Global", reinit=True)
wandb.config.update({
    "model_type": "DLinear",
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "kernel_size": KERNEL_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "patience": PATIENCE,
    "max_series": MAX_SERIES,
    "train_windows": len(X_train),
    "val_windows": len(X_val),
})

best_val_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    for x, y, h, mean, std in train_loader:
        x, y, h = x.to(device), y.to(device), h.to(device)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = weighted_mae_loss(y_pred, y, h)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for x, y, h, mean, std in val_loader:
            x, y, h = x.to(device), y.to(device), h.to(device)
            y_pred = model(x)
            loss = weighted_mae_loss(y_pred, y, h)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    wandb.log({"epoch": epoch, "train_loss_norm": train_loss, "val_loss_norm": val_loss})

    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:3d} | train_loss (norm) = {train_loss:.4f} | val_loss (norm) = {val_loss:.4f}")

    # --- Early stopping ---
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})")
            break

# საუკეთესო checkpoint-ის აღდგენა საბოლოო შეფასებამდე
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best model (val_loss_norm={best_val_loss:.4f})")

wandb.log({"best_val_loss_norm": best_val_loss})


## 6. საბოლოო WMAE შეფასება (ორიგინალურ სკალაზე)

Loss ტრენინგისას ნორმალიზებულ სკალაზეა (instance normalization-ის გამო) — საბოლოო
შედარებადი WMAE-სთვის საჭიროა **denormalization**, `mean`/`std`-ის უკან გამრავლებით,
სანამ WMAE-ს დავითვლით.

In [ ]:
model.eval()
all_preds, all_true, all_holidays = [], [], []

with torch.no_grad():
    for x, y, h, mean, std in val_loader:
        x = x.to(device)
        y_pred_norm = model(x).cpu().numpy()

        mean_np = mean.numpy().reshape(-1, 1)
        std_np = std.numpy().reshape(-1, 1)

        y_pred = y_pred_norm * std_np + mean_np
        y_true = y.numpy() * std_np + mean_np

        all_preds.append(y_pred)
        all_true.append(y_true)
        all_holidays.append(h.numpy())

all_preds = np.concatenate(all_preds)      # (n_val_windows, HORIZON)
all_true = np.concatenate(all_true)        # (n_val_windows, HORIZON)
all_holidays = np.concatenate(all_holidays).astype(bool)  # (n_val_windows, HORIZON)

# val_loader-ს shuffle=False აქვს, ამიტომ row-ების თანმიმდევრობა val_series_ids-ს ემთხვევა.
assert len(all_preds) == len(val_series_ids), "Row count mismatch — შეამოწმეთ val_loader-ის shuffle პარამეტრი"

dlinear_wmae_aggregate = wmae(all_true.flatten(), all_preds.flatten(), all_holidays.flatten())
print(f"DLinear Aggregate WMAE ({len(val_series_ids)} series evaluated together): {dlinear_wmae_aggregate:.2f}")
print("(Warning: This value is dominated by high-volume time series and is therefore not directly")
print(" comparable to the WMAE reported for individual ARIMA/SARIMA/Prophet runs.)")

wandb.log({"wmae_aggregate": dlinear_wmae_aggregate})


### Per-series WMAE breakdown

ზემოთ დათვლილი აგრეგირებული WMAE ერთ რიცხვში აერთიანებს ყველა სერიას (მაღალი
მოცულობის სერიები დომინირებენ, რადგან WMAE დოლარებშია). აქ ვშლით სერიების მიხედვით,
რომ (ა) ვნახოთ საშუალო/მედიანური per-series სირთულე, და (ბ) თუ `Store 1/Dept 1`
და `Store 20/Dept 92` შედის `MAX_SERIES` შერჩევაში — პირდაპირ შევადაროთ ARIMA/SARIMA/
Prophet-ის იმავე სერიების შედეგებს.

In [ ]:
# Per-series WMAE — პირდაპირი შედარებისთვის ARIMA/SARIMA/Prophet-ის იმავე 2 სერიასთან
per_series_wmae = {}
for i, sid in enumerate(val_series_ids):
    per_series_wmae[sid] = wmae(all_true[i], all_preds[i], all_holidays[i])

per_series_df = pd.DataFrame({
    "series_id": list(per_series_wmae.keys()),
    "wmae": list(per_series_wmae.values()),
}).sort_values("wmae")

print(f"Mean per-series WMAE ({len(per_series_df)} series): {per_series_df['wmae'].mean():.2f}")
print(f"Median: {per_series_df['wmae'].median():.2f}")

# პირდაპირი შედარება ARIMA/SARIMA/Prophet-ის იმავე 2 სერიასთან (თუ MAX_SERIES-ში შედის)
target_series = ["1_1", "20_92"]
for sid in target_series:
    if sid in per_series_wmae:
        print(f"DLinear — Store/Dept {sid}: WMAE = {per_series_wmae[sid]:.2f}")
        wandb.log({f"wmae_series_{sid}": per_series_wmae[sid]})
    else:
       print(
    f"Store/Dept {sid} is not included in the selected MAX_SERIES={MAX_SERIES} subset — "
    f"increase MAX_SERIES or verify the series_id ordering for a direct comparison."
)
run.finish()

per_series_df.head(10)
